In [3]:
!pip install earthengine-api geemap -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 61.3 MB/s eta 0:00:00


In [4]:
import ee
import geemap

ee.Authenticate()
ee.Initialize(project='sacred-store-468008-c8')
print("Earth Engine Initialized Successfully")
multan_aoi=ee.Geometry.Rectangle([71.35,30.10,71.60,30.30])
Map=geemap.Map(basemap='HYBRID')
Map.centerObject(multan_aoi,11)
Map.addLayer(multan_aoi,{},"Multan AOI Boundary")

Map

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


Earth Engine Initialized Successfully


Map(center=[30.200025418999658, 71.47499999999998], controls=(WidgetControl(options=['position', 'transparent_…

In [5]:
def get_sentinel2_data(year,aoi):
  start_date=f'{year}-01-01'
  end_date=f'{year}-12-31'

  collection=(ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(aoi).filterDate(start_date,end_date).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE',20)))
  return collection.median().clip(aoi)


print("Satellite Images are fetching.Kindly be wait!")
image_2019=get_sentinel2_data(2019,multan_aoi)
image_2025=get_sentinel2_data(2025,multan_aoi)
rgb_vis={
    'bands':['B4','B3','B2'],
    'min':0,
    'max':3000,
    'gamma':1.4
}
Map.addLayer(image_2019,rgb_vis,"Multan 2019(RGB)",False)
Map.addLayer(image_2025,rgb_vis,"Multan 2025(RGB)",True)
print("Images are adding on the map")
Map





Satellite Images are fetching.Kindly be wait!
Images are adding on the map


Map(bottom=216272.0, center=[30.200025418999658, 71.47499999999998], controls=(WidgetControl(options=['positio…

In [7]:
def add_indices(image):
  ndvi=image.normalizedDifference(['B8','B4']).rename('NDVI')
  ndbi=image.normalizedDifference(['B11','B8']).rename('NDBI')
  return image.addBands([ndvi,ndbi])

image_2019=add_indices(image_2019)
image_2025=add_indices(image_2025)
ndvi_vis={
    'min':0.0,
    'max':0.8,
    'paletter':['white','lightgreen','darkgreen']
}
ndbi_vis={
    'min':-0.2,
    'max':0.4,
    'palette':['white','orange','red']
}
Map.addLayer(image_2019.select('NDVI'),ndvi_vis,'Multan 2019(NDVI)',False)
Map.addLayer(image_2025.select('NDVI'),ndvi_vis,'Multan 2025(NDVI)',True)
Map.addLayer(image_2019.select('NDBI'),ndbi_vis,'Multan 2019(NDBI)',False)
Map.addLayer(image_2025.select('NDBI'),ndbi_vis,'Multan 2025(NDBI)',True)
print("MDVI and NDBI are successfully calculated")
Map


MDVI and NDBI are successfully calculated


Map(bottom=216272.0, center=[30.200025418999658, 71.47499999999998], controls=(WidgetControl(options=['positio…

In [8]:
import ee
import geemap

Map=geemap.Map(basemap='HYBRID')
Map.centerObject(multan_aoi,11)

def add_advance_indices(image):
  ndvi=image.normalizedDifference(['B8','B4']).rename('NDVI')
  ndbi=image.normalizedDifference(['B11','B8']).rename('NDBI')
  ndwi=image.normalizedDifference(['B3','B8']).rename('NDWI')
  return image.addBands([ndvi,ndbi,ndwi])


image_2019_adv=add_advance_indices(image_2019)
image_2025_adv=add_advance_indices(image_2025)
prediction_bands=['B2','B3','B4','B8','B11','NDVI','NDBI','NDWI']
advance_rules=ee.Image(4).where(image_2019_adv.select('NDVI').gt(0.2),0).where(image_2019_adv.select('NDWI').gt(0),2).where(image_2019_adv.select('NDBI').gt(0).And(image_2019_adv.select('NDWI').lte(0)),1).where(image_2019_adv.select('NDVI').lt(0.1).And(image_2019_adv.select('NDBI').lt(0)),3).rename('class')
training_image=image_2019_adv.select(prediction_bands).addBands(advance_rules)
training_samples=training_image.stratifiedSample(
    numPoints=50,
    classBand='class',
    region=multan_aoi,
    scale=10,
    geometries=True
)
print("Advance Random Forest Model is training...(5 classes)...")
rf_classifier=ee.Classifier.smileRandomForest(100).train(
    features=training_samples,
    classProperty='class',
    inputProperties=prediction_bands
)
classified_2019=image_2019_adv.select(prediction_bands).classify(rf_classifier)
classified_2025=image_2025_adv.select(prediction_bands).classify(rf_classifier)
class_vis={
    'min':0,
    'max':4,
    'palette':['green','red','blue','brown','yellow']
}
Map.addLayer(classified_2019,class_vis,'Adv Classified 2019',True)
Map.addLayer(classified_2025,class_vis,'Adv Classified 2025',True)
print("Advance Classification complete. Kindly check layers on map")
Map



Advance Random Forest Model is training...(5 classes)...
Advance Classification complete. Kindly check layers on map


Map(center=[30.200025419001395, 71.47499999999998], controls=(WidgetControl(options=['position', 'transparent_…

In [9]:
import pandas as pd

veg_to_urban_change=classified_2019.eq(0).And(classified_2025.eq(1))
Map.addLayer(veg_to_urban_change.selfMask(),{'palette':['magenta']},'Agricultural to Urban Conversion',True)
print("Area is calculating... Kindly be patience!")
def calculate_area(image_mask,region):
  area=ee.Image.pixelArea().updateMask(image_mask).reduceRegion(
      reducer=ee.Reducer.sum(),
      geometry=region,
      scale=10,
      maxPixels=1e10
  ).get('area')

  return ee.Number(area).divide(1e6)

total_aoi_mask=ee.Image(1).clip(multan_aoi)
veg_2019_mask=classified_2019.eq(0)
builtup_2025_mask=classified_2025.eq(1)

total_area_sqkm=calculate_area(total_aoi_mask,multan_aoi).getInfo()
veg_area_2019=calculate_area(veg_2019_mask,multan_aoi).getInfo()
builtup_area_2025=calculate_area(builtup_2025_mask,multan_aoi).getInfo()
converted_area=calculate_area(veg_to_urban_change,multan_aoi).getInfo()
percentage_converted=(converted_area/total_area_sqkm)*100
data={
    "Metric":[
        "Total AOI Area",
        "Vegetation Area in 2019",
        "Built-up Area in 2025",
        "Agriculture-to-Urban Conversion",
        "Percentage Converted"
    ],
    "Value":[
        f"{total_area_sqkm:.2f} sq km",
        f"{veg_area_2019:.2f} sq km",
        f"{builtup_area_2025:.2f} sq km",
        f"{converted_area:.2f} sq km",
        f"{percentage_converted:.2f} %"
    ]
}
df=pd.DataFrame(data)
print("\nArea Statistics\n")
print(df.to_string(index=False))
Map



Area is calculating... Kindly be patience!

Area Statistics

                         Metric        Value
                 Total AOI Area 533.73 sq km
        Vegetation Area in 2019 288.38 sq km
          Built-up Area in 2025 206.32 sq km
Agriculture-to-Urban Conversion  24.22 sq km
           Percentage Converted       4.54 %


Map(bottom=216272.0, center=[30.200025419001395, 71.47499999999998], controls=(WidgetControl(options=['positio…

In [10]:
print("Calculating Model Accuracy...")
train_accuracy=rf_classifier.confusionMatrix()
overall_accuracy=train_accuracy.accuracy().getInfo()
print(f"Model overall Accuracy: {overall_accuracy * 100:.2f}%\n")
print("Confusion Matrix: ")
print(train_accuracy.getInfo())
print("\nExporting Final Change map...")
export_task=ee.batch.Export.image.toDrive(
    image=veg_to_urban_change.toInt(),
    description='Multan_Urban_Expansion_2019_2025',
    folder='EarthEngine_Projects',
    region=multan_aoi,
    scale=10,
    maxPixels=1e10
)
export_task.start()
print("Export task background start...")


Calculating Model Accuracy...
Model overall Accuracy: 99.60%

Confusion Matrix: 
[[50, 0, 0, 0, 0], [1, 49, 0, 0, 0], [0, 0, 50, 0, 0], [0, 0, 0, 50, 0], [0, 0, 0, 0, 50]]

Exporting Final Change map...
Export task background start...


In [11]:
!pip install -q streamlit geemap earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 106.0 MB/s eta 0:00:00


In [25]:
%%writefile app.py
import streamlit as st
import folium
import streamlit.components.v1 as components
import ee
import requests

st.set_page_config(layout="wide",page_title="GeoGuardian Dashboard")
st.title("GeoGuardian: Multan Urban Expansion Monitor")
st.markdown("Monitoring agricultural land loss and urban builtup expansion using Sentinel-2 and Rnadom Forest")


try:
    ee.Initialize(project='sacred-store-468008-c8')
except Exception as e:
  import google.auth
  credentials,project=google.auth.default()
  ee.Initialize(credentials,project='sacred-store-468008-c8')



st.subheader("Change Detection Statistics (2019-2025)")
col1,col2,col3,col4=st.columns(4)
with col1:
  st.info("**Total Aoi**\n### 533.73 sq km")
with col2:
  st.error("**Agriculture Lost**\n### 24.22 sq km")
with col3:
  st.warning("**Percentage Converted**\n### 4.54%")
with col4:
  st.success("**Builtup 2025**\n### 206.32 sq km")

st.divider()
st.subheader("Interactive Satellite view")
# m=geemap.Map(basemap='HYBRID')
multan_aoi=ee.Geometry.Rectangle([71.35,30.10,71.60,30.30])
m=folium.Map(location=[30.2,71.45],zoom_start=11)
def add_ee_layer(self,ee_image_object,vis_params,name):
  map_id_dict=ee.Image(ee_image_object).getMapId(vis_params)
  folium.raster_layers.TileLayer(
      tiles=map_id_dict['tile_fetcher'].url_format,
      attr='Map Data Google Earth Engine',
      name=name,
      overlay=True,
      control=True
  ).add_to(self)

folium.Map.add_ee_layer=add_ee_layer
empty=ee.Image().byte()
aoi_outline=empty.paint(featureCollection=multan_aoi,color=1,width=3)
m.add_ee_layer(aoi_outline,{'palette':'red'},'Multan AOI Boundary')
folium.LayerControl(collapsed=False).add_to(m)
components.html(m._repr_html_(),height=600)
# m.to_streamlit(height=600)

Overwriting app.py


In [26]:
!npm install localtunnel
import urllib

print("Password/Endpoint IP for localtunnel is: ",urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
!streamlit run app.py & npx localtunnel --port 8501

⠙⠹⠸⠼
up to date, audited 23 packages in 707ms
⠼
⠼3 packages are looking for funding
⠼  run `npm fund` for details
⠼
2 high severity vulnerabilities

To address all issues (including breaking changes), run:
  npm audit fix --force

Run `npm audit` for details.
⠼Password/Endpoint IP for localtunnel is:  34.16.156.82
⠙

your url is: https://smart-banks-float.loca.lt
2026-09-24 09:20:41.414 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.16.156.82:8501

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python
2026-09-24 09:21:15.065 Please replace `st.components.v1.html` with `st.iframe`.

`st.components.v1.html` will be removed after 2026-06-01.
  Stopping...
^C


In [ ]:
!pip install -q streamlit streamlit-folium folium earthengine-api

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.9/536.9 kB 11.8 MB/s eta 0:00:00


In [1]:
%%writefile app.py
import streamlit as st
import ee
import folium
import streamlit.components.v1 as components

st.set_page_config(layout="wide",page_title="GeoGuardian Dashboard")
st.title("GeoGuardian: Multan Urban Expansion Monitor")
st.markdown("Build By Muhammad Khurram-AI/ML Intern")
st.markdown("Monitoring agricultural land loss and urban built-up expansion using Sentinel-2 and Random Forest.")

try:
    ee.Initialize(project='sacred-store-468008-c8')
except Exception as e:
    import google.auth
    credentials,project=google.auth.default()
    ee.Initialize(credentials,project='sacred-store-468008-c8')

st.subheader("Change Detection Statistics (2019 - 2025)")
col1,col2,col3,col4=st.columns(4)
with col1:
  st.info("**Total AOI**\n### 533.73 sq km")
with col2:
  st.error("**Agriculture Lost**\n### 24.22 sq km")
with col3:
  st.warning("**Percentage Converted**\n### 4.54%")
with col4:
  st.success("**Built-up 2025**\n### 206.32 sq km")

st.divider()

st.subheader("Interactive Satellite View")
multan_aoi=ee.Geometry.Rectangle([71.35,30.10,71.60,30.30])
m=folium.Map(location=[30.2,71.45],zoom_start=11)

def add_ee_layer(self,ee_image_object,vis_params,name):
    map_id_dict=ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles=map_id_dict['tile_fetcher'].url_format,
        attr='Map Data Google Earth Engine',
        name=name,
        overlay=True,
        control=True
    ).add_to(self)
folium.Map.add_ee_layer=add_ee_layer

empty=ee.Image().byte()
aoi_outline=empty.paint(featureCollection=multan_aoi,color=1,width=3)
m.add_ee_layer(aoi_outline,{'palette':'red'},'Multan AOI Boundary')
folium.LayerControl(collapsed=False).add_to(m)

components.html(m._repr_html_(),height=600)

Writing app.py


In [2]:
!npm install -g localtunnel
import urllib
print("Password/Endpoint IP is:",urllib.request.urlopen('https://ipv4.icanhazip.com').read().decode('utf8').strip("\n"))
!streamlit run app.py & lt --port 8501

⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏
added 22 packages in 3s
⠏
⠏3 packages are looking for funding
⠏  run `npm fund` for details
⠏Password/Endpoint IP is: 34.16.156.82
/bin/bash: line 1: streamlit: command not found
your url is: https://sour-flowers-dream.loca.lt
^C
